## 5. Confound Check: Recording Bandwidth

The GTZAN excerpts were derived from lossy, band-limited sources: the mean rock spectrum falls steeply above about 9 kHz, while the AdP recordings extend to 11 kHz. Timbre features describe the shape of the spectrum, so a classifier could in principle separate the classes by this recording artefact rather than by anything musical.

Two checks address this:
1. **The band above 9 kHz on its own.** If this single value separates the classes, the artefact is real and available to any model.
2. **Timbre features on low-passed audio.** Both classes are filtered at 8 kHz, removing the band entirely, and the timbre features are recomputed. If performance stays close to the full-band result, the timbre separation does not depend on the codec cutoff.

In [ ]:
from scipy.signal import butter, sosfiltfilt

# GTZAN clips come from lossy, band-limited sources: the mean rock spectrum falls
# steeply above ~8.5 kHz, while the AdP masters extend to 11 kHz. Low-passing BOTH
# classes at the same frequency removes that band, so whatever separation survives
# cannot come from the codec cutoff.
LOWPASS_HZ = 8000      # filter cutoff, safely below the GTZAN roll-off
HF_DIAG_HZ = 9000      # band where the GTZAN roll-off actually shows (diagnostic only)
_SOS = butter(8, LOWPASS_HZ, btype="low", fs=SR, output="sos")


def lowpass(y):
    """Zero-phase 8th-order Butterworth low-pass at LOWPASS_HZ."""
    return sosfiltfilt(_SOS, y)


def hf_energy_db(y, f_lo=HF_DIAG_HZ):
    """Share of total energy above f_lo, in dB. A direct measure of the codec cutoff."""
    S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP)) ** 2
    freqs = librosa.fft_frequencies(sr=SR, n_fft=N_FFT)
    return float(10 * np.log10(S[freqs >= f_lo].sum() / (S.sum() + 1e-12) + 1e-12))


def build_lowpass_timbre(manifest, cache=CACHE / "timbre_lowpass.csv", force=False):
    """Timbre features recomputed on low-passed audio, plus the high-frequency energy
    of the original audio. Cached separately; no pYIN, so it runs in a few minutes."""
    if cache.exists() and not force:
        print(f"Loaded cached low-pass features from {cache}")
        return pd.read_csv(cache)

    rows = []
    for n, r in enumerate(manifest.itertuples(), 1):
        if n % 20 == 0 or n == 1:
            print(f"  {n}/{len(manifest)}")
        y = load_clip(r.path)
        f_lp = timbre_features(lowpass(y))
        rows.append({"clip_id": r.clip_id,
                     "conf_hf_energy_db": hf_energy_db(y),
                     **{f"{k}_lp": v for k, v in f_lp.items()}})

    lp = pd.DataFrame(rows)
    lp.to_csv(cache, index=False)
    print(f"Saved {lp.shape[0]} x {lp.shape[1]} to {cache}")
    return lp


lp = build_lowpass_timbre(manifest).set_index("clip_id").loc[features["clip_id"]]
features[lp.columns] = lp.values   # same row order as `features`, safe to re-run

TIMBRE_LP_COLS = [c + "_lp" for c in FEATURE_SETS["timbre"]]
print(f"{len(TIMBRE_LP_COLS)} low-passed timbre features added")

In [ ]:
# Three questions:
#  1. How much does the band above 9 kHz alone reveal the class?
#  2. Does timbre still separate the classes once that band is removed?
#  3. Does the full-band result change when the same features are low-passed?
confound_sets = {
    "energy above 9 kHz only":         ["conf_hf_energy_db"],
    "timbre, full band":               FEATURE_SETS["timbre"],
    f"timbre, low-passed at {LOWPASS_HZ // 1000} kHz": TIMBRE_LP_COLS,
}

rows = []
for set_name, cols in confound_sets.items():
    for model_name, model in make_models().items():
        if model_name == "Majority baseline":
            continue
        r = evaluate(model, cols)
        rows.append({"features": set_name, "model": model_name,
                     "accuracy": r["accuracy"], "f1": r["f1"], "f1_std": r["f1_std"]})

confound_results = pd.DataFrame(rows)

print("Energy above 9 kHz, median per class (dB relative to total):")
print(features.groupby("label_name")["conf_hf_energy_db"].median().round(1).to_string())

confound_results.round(3)

#### Interpretation

**The bandwidth artefact is real but does not explain the result.** Energy above 9 kHz is 8 dB lower in the GTZAN clips than in the AdP recordings (median −38.5 vs −30.6 dB), and this single value alone reaches F1 = 0.84 with the SVM. A classifier could therefore exploit the codec cutoff. However, timbre features computed on audio low-passed at 8 kHz, where the band is absent in both classes, perform as well as the full-band features (F1 0.96–0.97 vs 0.96–0.98, within the fold-to-fold spread). The timbral separation therefore lies below 8 kHz, in the instrumentation and playing style rather than in the recording bandwidth.

This check rules out only one recording difference. Others remain: AdP is a single artist with a consistent studio production and modern mastering, while GTZAN spans many artists, studios and decades. With one artist in the positive class, the classifier may partly be learning to recognise this band's sound rather than a general musical style, and the present data cannot separate the two.